In [3]:
import json
import random


films = []
with open('full_dump.jsonl', 'r', encoding='UTF-8') as f:
    for line in f:
        films.append(json.loads(line))

n = len(films)
print(n)

847209


In [4]:
from langdetect import detect
import json
import random

annotations = []
viewed = [0]
count = 0
movie_subset = 100000
with open("top_reviews.jsonl","w",encoding='UTF-8') as f:
    while count < movie_subset:
        r = random.randint(0,n-1)
        current_film = films[r]
        reviews_list = current_film.get("reviews", [])


        if not reviews_list:
            continue 
        try:
            if detect(current_film.get("reviews")[0].get("review_text")) != 'en':
                continue
        except:
            continue
        if current_film.get('rating') is None:
            continue
        if current_film.get("reviews")[0].get("review_text").strip().endswith("...") or current_film.get("reviews")[0].get("review_text").strip().endswith("…"):
            continue
        data = {
            "title" : current_film.get("title",),
            "year" : current_film.get("year"),
            "synopsis" : current_film.get("synopsis"),
            "reviews" : current_film.get("reviews")[0].get("review_text"),
            "rating" : current_film.get('rating')
        }
        f.write(json.dumps(data) + "\n")
        count +=1

In [5]:

random_films = []
with open('top_reviews.jsonl', 'r', encoding='UTF-8') as f:
    for line in f:
        random_films.append(json.loads(line))


In [6]:
import sklearn
train_set, test_set = sklearn.model_selection.train_test_split(random_films, test_size=0.1)
train_texts = []
train_labels = []
test_texts = []
test_labels = []
for item in train_set:
    train_texts.append(item['reviews'])
    train_labels.append(round(float(item['rating'][:4])))
for item in test_set:
    test_texts.append(item['reviews'])
    test_labels.append(round(float(item['rating'][:4])))


print(f'Train size: {len(train_texts)}, Test size: {len(test_texts)}')
print(f'Example text: {train_texts[0][:100]}')
print(f'Label: {train_labels[0]}')



Train size: 90000, Test size: 10000
Example text: So I saw a dark stranger and I found it compelling for it's somewhat comedic narrative that is mainl
Label: 3


# Logistic Regression Classifier

In [12]:
# --- Bag-of-Words Model ---
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
# YOUR CODE HERE: vectorize texts using CountVectorizer
vectorizer = CountVectorizer()  # TODO

X_train = vectorizer.fit_transform(train_texts)  # TODO
X_test  = vectorizer.transform(test_texts)  # TODO


clf_bow = LogisticRegression(max_iter=10000) # TODO
clf_bow.fit(X_train, train_labels)


y_pred_bow = clf_bow.predict(X_test)  # TODO

acc_bow = accuracy_score(test_labels, y_pred_bow)
f1_bow  = f1_score(test_labels, y_pred_bow, average='macro')
precision = precision_score(test_labels,y_pred_bow, average='macro')
recall = recall_score(test_labels,y_pred_bow, average='macro')
print(f'Accuracy: {acc_bow:.4f}')
print(f'Macro F1: {f1_bow:.4f}')
print(f'Precision:{precision:.4f}')
print(f'Recall:{recall:.4f}')


Accuracy: 0.8117
Macro F1: 0.6053
Precision:0.8838
Recall:0.5156


In [10]:
import pickle
pickle.dump(clf_bow, open('logistic_regression_classifier','wb'))

In [ ]:
model = pickle.load(open('logistic_regression_classifier','rb'))
